In [ ]:
# ==============================================================================
# MACHINE LEARNING PIPELINE: SEOUL BIKE DEMAND PREDICTION
# ==============================================================================

# --- IMPORTS ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Starting Machine Learning Pipeline...")

# ==============================================================================
# STEP 1: LOAD AND CLEAN THE BASE DATA
# ==============================================================================

# 1. Load the data (using latin-1 to handle special characters like the degree symbol)
# NOTE: Make sure the file path is correct for your computer!
df = pd.read_csv('data/raw/SeoulBikeData.csv', encoding='latin-1')

# 2. Rename columns to plain English so we don't get formatting errors later
df.columns = ['Date', 'Rented_Bike_Count', 'Hour', 'Temperature', 'Humidity', 
              'Wind_Speed', 'Visibility', 'Dew_Point', 'Solar_Radiation', 
              'Rainfall', 'Snowfall', 'Seasons', 'Holiday', 'Functioning_Day']

# 3. Drop days where the bike system was physically closed
df = df[df['Functioning_Day'] == 'Yes']
df = df.drop('Functioning_Day', axis=1)

# ==============================================================================
# STEP 2: GLOBAL FEATURE ENGINEERING
# ==============================================================================

# 1. Create a custom "Weather_Condition" feature to combine noisy weather data
def categorize_weather(row):
    if row['Rainfall'] > 0 or row['Snowfall'] > 0 or row['Wind_Speed'] > 5 or row['Visibility'] < 500:
        return 'Bad'
    elif row['Rainfall'] == 0 and row['Snowfall'] == 0 and row['Wind_Speed'] <= 3 and row['Visibility'] >= 1500:
        return 'Good'
    else:
        return 'Regular'

# Apply the function row-by-row
df['Weather_Condition'] = df.apply(categorize_weather, axis=1)

# 2. Extract standard numbers from the Date string
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df = df.drop('Date', axis=1) # Drop the original text date

print("Base Data Cleaning and Feature Engineering Complete.")

# ==============================================================================
# STEP 3: CREATE THE 4 EXPERIMENTAL DATABASES
# ==============================================================================
databases = {} # We will store our Train/Test splits in this dictionary

# ---------------------------------------------------------
# DATABASE 1: Raw Data + Dummies
# (No custom weather, no cyclical dates, no scaling)
# ---------------------------------------------------------
df1 = df.drop(['Weather_Condition', 'Month', 'DayOfWeek'], axis=1)
# Convert text categories to 1s and 0s
df1 = pd.get_dummies(df1, columns=['Seasons', 'Holiday'], drop_first=True)

X1 = df1.drop('Rented_Bike_Count', axis=1)
y1 = df1['Rented_Bike_Count']
databases["1. Raw Data (Basic Dummies)"] = train_test_split(X1, y1, test_size=0.2, random_state=42)


# ---------------------------------------------------------
# DATABASE 2: Tree Data (Custom Weather + Cyclical Dates, UNSCALED)
# ---------------------------------------------------------
df2 = df.copy()
# Drop redundant weather columns since we made 'Weather_Condition'
df2 = df2.drop(['Rainfall', 'Snowfall', 'Dew_Point'], axis=1)
# Convert text categories to 1s and 0s
df2 = pd.get_dummies(df2, columns=['Seasons', 'Holiday', 'Weather_Condition'], drop_first=True)

# Create "Clock Face" cyclical dates for math models
df2['Month_sin'] = np.sin(2 * np.pi * df2['Month'] / 12)
df2['Month_cos'] = np.cos(2 * np.pi * df2['Month'] / 12)
df2['DayOfWeek_sin'] = np.sin(2 * np.pi * df2['DayOfWeek'] / 7)
df2['DayOfWeek_cos'] = np.cos(2 * np.pi * df2['DayOfWeek'] / 7)
df2 = df2.drop(['Month', 'DayOfWeek'], axis=1) # Drop the old flat numbers

X2 = df2.drop('Rented_Bike_Count', axis=1)
y2 = df2['Rented_Bike_Count']
databases["2. Tree Data (Engineered, Cyclical, Unscaled)"] = train_test_split(X2, y2, test_size=0.2, random_state=42)


# ---------------------------------------------------------
# DATABASE 3: Regression Data (Engineered + Cyclical, SCALED)
# ---------------------------------------------------------
# Database 3 is identical to Database 2, but we apply Standard Scaling to the numbers
X3_train, X3_test, y3_train, y3_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

# We ONLY scale continuous numbers, not the 1s and 0s or the cyclical Sine/Cosine
num_cols = ['Hour', 'Temperature', 'Humidity', 'Wind_Speed', 'Visibility', 'Solar_Radiation']
scaler3 = StandardScaler()

X3_train[num_cols] = scaler3.fit_transform(X3_train[num_cols])
X3_test[num_cols] = scaler3.transform(X3_test[num_cols])

databases["3. Regression Data (Engineered, Cyclical, SCALED)"] = (X3_train, X3_test, y3_train, y3_test)


# ---------------------------------------------------------
# DATABASE 4: Cleaned Regression Data (Engineered, Scaled, OUTLIERS REMOVED)
# ---------------------------------------------------------
df4 = df2.copy() # Start with the Database 2 structure

# Identify and remove extreme spikes in Bike Rentals using the IQR Boxplot math
Q1 = df4['Rented_Bike_Count'].quantile(0.25)
Q3 = df4['Rented_Bike_Count'].quantile(0.75)
IQR = Q3 - Q1
df4 = df4[(df4['Rented_Bike_Count'] >= (Q1 - 1.5 * IQR)) & (df4['Rented_Bike_Count'] <= (Q3 + 1.5 * IQR))]

X4 = df4.drop('Rented_Bike_Count', axis=1)
y4 = df4['Rented_Bike_Count']
X4_train, X4_test, y4_train, y4_test = train_test_split(X4, y4, test_size=0.2, random_state=42)

# Scale the continuous numbers just like in Database 3
scaler4 = StandardScaler()
X4_train[num_cols] = scaler4.fit_transform(X4_train[num_cols])
X4_test[num_cols] = scaler4.transform(X4_test[num_cols])

databases["4. Regression Data (Scaled, Target Outliers Removed)"] = (X4_train, X4_test, y4_train, y4_test)

print("All 4 Databases successfully created!")

# ==============================================================================
# STEP 4: THE MASTER EVALUATION LOOP
# ==============================================================================

# Define the models we want to test
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

print("\nBeginning Model Evaluation Tournament...")

# 1. Loop through each dataset
for db_name, (X_train, X_test, y_train, y_test) in databases.items():
    print(f"\n{'='*65}")
    print(f" TESTING: {db_name}")
    print(f"{'='*65}")
    
    # 2. Loop through each model for the current dataset
    for model_name, model in models.items():
        
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        preds = model.predict(X_test)
        
        # Calculate evaluation scores
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)
        
        # Print the final results
        print(f"[{model_name}]")
        print(f"  -> RMSE: {rmse:.1f} bikes off average")
        print(f"  -> MAE:  {mae:.1f} bikes off median")
        print(f"  -> R² Score: {r2:.4f} (Closer to 1.0 is better)")
        print("-" * 30)

print("\nPipeline Complete! Review your results above.")